# Step 1: Skull stripping with HD-BET

Original Kaggle notebook used for the thesis experiments (outputs cleared; code unchanged).

- **Input:** raw MS brain MRI volumes (`.nii`) from the public Mendeley dataset (Muslim et al., 2022), as uploaded to Kaggle.
- **Output:** skull-stripped volumes (`*_bet.nii.gz`) in `data_processed_bet/`.
- Lesion masks (paths containing `LesionSeg`) are skipped. HD-BET runs on GPU with test-time augmentation disabled.
- A cleaned command-line version of this step is in `preprocessing/skull_strip.py`.
- The Kaggle input path is specific to the original environment and must be changed to run elsewhere.


In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install hd-bet

In [ ]:
!hd-bet --force_download

In [ ]:
# ZIP_PATH = '/kaggle/input/multiplesclerosis-60-patients-data'
RAW_DIR = '/kaggle/input/multiplesclerosis-60-patients-data'

In [ ]:
# !mkdir -p $EXTRACT_DIR
# !unzip -q $ZIP_PATH -d $EXTRACT_DIR

In [ ]:
import os

PROCESSED_DIR = 'data_processed_bet'
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Setup Complete. Input Path set to: {RAW_DIR}")
print(f"Output Path set to: {PROCESSED_DIR}")

In [ ]:
%%bash

# --- 1. Define Variables for the Shell Script ---
# Use the correct Kaggle path
RAW_DIR='/kaggle/input/multiplesclerosis-60-patients-data' # Input Path
PROCESSED_DIR='data_processed_bet' # Output Path
 
# --- 2. Find Files and Loop ---
# Searches recursively through the entire dataset folder for all .nii files.
NII_FILES=$(find $RAW_DIR -type f -name "*.nii")

echo "Found $(echo $NII_FILES | wc -w) NII files for processing."

# Loop and run HD-BET for each file
for INPUT_FILE in $NII_FILES; do
    
    # Skip binary Lesion Segmentation masks
    if [[ "$INPUT_FILE" == *"LesionSeg"* ]]; then
        echo "Skipping segmentation mask: $INPUT_FILE"
        continue
    fi

    # Extract filename and create output path
    FILENAME=$(basename "$INPUT_FILE")
    BASE_NAME=${FILENAME%.nii}
    OUTPUT_FILE="$PROCESSED_DIR/${BASE_NAME}_bet.nii.gz"

    echo "Processing $INPUT_FILE to $OUTPUT_FILE"

    # Run HD-BET using the GPU (cuda)
    hd-bet -i "$INPUT_FILE" -o "$OUTPUT_FILE" -device cuda --disable_tta
done

echo "--- All anatomical scans have been processed. The skull-stripped files (_bet.nii.gz) are in the '$PROCESSED_DIR' folder. ---"